In [126]:
from dotenv import load_dotenv
from dataclasses import dataclass
from langchain_community.utilities import SQLDatabase
from langchain_community.utilities import SQLDatabase
from langchain_core.tools import tool
from langgraph.runtime import get_runtime
from langchain.agents import create_agent
from langchain_ollama import ChatOllama
from langchain_google_genai import ChatGoogleGenerativeAI


In [127]:
load_dotenv()

True

In [128]:

db = SQLDatabase.from_uri("sqlite:///Chinook.db")

@dataclass
class RuntimeContext:
    db: SQLDatabase

In [129]:
@tool
def execute_sql(query: str) -> str:
    """Execute a SQLite command and return results."""
    runtime = get_runtime(RuntimeContext)
    db = runtime.context.db

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

In [130]:
SYSTEM_PROMPT = """You are a careful SQLite analyst.

Rules:
- Think step-by-step.
- Call execute_sql as many times as needed to answer the question.
- Read-only only; no INSERT/UPDATE/DELETE/ALTER/DROP/CREATE/REPLACE/TRUNCATE.
- Limit to 50 rows of output unless the user explicitly asks otherwise.
- If the tool returns 'Error:', revise the SQL and try again.
- Prefer explicit column lists; avoid SELECT *.
"""

In [ ]:
#llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")


agent = create_agent(
    model= ChatOllama(
    model="qwen3.5:9b",
    temperature=0,
    num_ctx=8192,
    num_predict=-1,
    reasoning=False, 
),
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,

)

In [132]:
question = "Which table has the largest number of entries?"
for step in agent.stream(
    {"messages": question},
    context=RuntimeContext(db=db),
    stream_mode="values",

):
    step["messages"][-1].pretty_print()
    



================================ Human Message =================================

Which table has the largest number of entries?


================================== Ai Message ==================================
Tool Calls:
  execute_sql (04bd6236-84bf-437f-9954-277a7e2a81dc)
 Call ID: 04bd6236-84bf-437f-9954-277a7e2a81dc
  Args:
    query: SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;
================================= Tool Message =================================
Name: execute_sql

[('Album',), ('Artist',), ('Customer',), ('Employee',), ('Genre',), ('Invoice',), ('InvoiceLine',), ('MediaType',), ('Playlist',), ('PlaylistTrack',), ('Track',)]
================================== Ai Message ==================================
